In [ ]:
import json
import time
import boto3
import sagemaker

from sagemaker import ModelPackage
from sagemaker import get_execution_role

In [ ]:
role = get_execution_role()
sagemaker_session = sagemaker.Session()
smr_client = boto3.client("sagemaker-runtime")

model_package_name = "ixi-gen-fin-7-8b-250420-v1-775ffd5f8abd317e88506bf9281f92af"
model_package_map = {
    "us-east-1": f"arn:aws:sagemaker:us-east-1:865070037744:model-package/{model_package_name}",
    "us-east-2": f"arn:aws:sagemaker:us-east-2:057799348421:model-package/{model_package_name}",
    "us-west-1": f"arn:aws:sagemaker:us-west-1:382657785993:model-package/{model_package_name}",
    "us-west-2": f"arn:aws:sagemaker:us-west-2:594846645681:model-package/{model_package_name}",
    "ca-central-1": f"arn:aws:sagemaker:ca-central-1:470592106596:model-package/{model_package_name}",
    "eu-central-1": f"arn:aws:sagemaker:eu-central-1:446921602837:model-package/{model_package_name}",
    "eu-west-1": f"arn:aws:sagemaker:eu-west-1:985815980388:model-package/{model_package_name}",
    "eu-west-2": f"arn:aws:sagemaker:eu-west-2:856760150666:model-package/{model_package_name}",
    "eu-west-3": f"arn:aws:sagemaker:eu-west-3:843114510376:model-package/{model_package_name}",
    "eu-north-1": f"arn:aws:sagemaker:eu-north-1:136758871317:model-package/{model_package_name}",
    "ap-southeast-1": f"arn:aws:sagemaker:ap-southeast-1:192199979996:model-package/{model_package_name}",
    "ap-southeast-2": f"arn:aws:sagemaker:ap-southeast-2:666831318237:model-package/{model_package_name}",
    "ap-northeast-2": f"arn:aws:sagemaker:ap-northeast-2:745090734665:model-package/{model_package_name}",
    "ap-northeast-1": f"arn:aws:sagemaker:ap-northeast-1:977537786026:model-package/{model_package_name}",
    "ap-south-1": f"arn:aws:sagemaker:ap-south-1:077584701553:model-package/{model_package_name}",
    "sa-east-1": f"arn:aws:sagemaker:sa-east-1:270155090741:model-package/{model_package_name}",
}

region = sagemaker_session.boto_region_name
if region not in model_package_map.keys():
    raise Exception(f"Current boto3 session region {region} is not supported.")

model_pack_arn = model_package_map[region]
model = ModelPackage(
    role=role,
    model_package_arn=model_pack_arn,
    sagemaker_session=sagemaker_session
)

print(f"role: {role}")
print(f"region: {region}")
print(f"model package arn: {model_pack_arn}")

In [ ]:
model_name = "ixi-gen-fin-7-8b-250420-v1"

model.deploy(
    initial_instance_count=1,
    instance_type='ml.g5.4xlarge',
    endpoint_name=sagemaker.utils.name_from_base(model_name),
    model_data_download_timeout=600,
    container_startup_health_check_timeout=300,
)

print(f"endpoint name: {model.endpoint_name}")

In [ ]:
# 클라이언트 생성
sagemaker_runtime = boto3.client('sagemaker-runtime', region_name=region)

prompt = "who are you?"

# 요청 데이터 준비
request_data = {
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "max_tokens": 8192,
    "temperature": 0.0,
    "repetition_penalty": 1.0,
}

# 호출
response = sagemaker_runtime.invoke_endpoint(
    EndpointName=model.endpoint_name,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps(request_data),
)

# 결과 디코딩
result = response['Body'].read().decode('utf-8')
print("응답:", result)

In [ ]:
model.sagemaker_session.delete_endpoint(model.endpoint_name)
model.sagemaker_session.delete_endpoint_config(model.endpoint_name)
model.delete_model()

print(f'--- Deleted endpoint_config: {model.endpoint_name}')
client = boto3.client('sagemaker')
response = boto3.client('sagemaker').describe_endpoint(EndpointName=model.endpoint_name)
print(f'--- Deleted endpoint: {response["EndpointConfigName"]}')
print(f'--- Deleted model: {model.name}')